# SVM Models

In [12]:
# reading data sets
import pandas as pd
import numpy as np
import os


meta_df = pd.read_csv("data/processed_data/YACHIDA_CRC_2019/metadata.tsv", sep="\t")
meta_df['Study.Group'] = meta_df['Study.Group'].replace({'HS': 'Healthy'})
print(meta_df['Study.Group'].value_counts(dropna=False))

folder_path = r"C:\Users\tonyp\compSci\Research\Multimodal_Microbiome_Fusion_Project\Meta_Fusion\data\processed_data\YACHIDA_CRC_2019\Binary_datasets"

# diseased
MGX_df_diseased = pd.read_csv(
    os.path.join(folder_path, "MGX_df_diseased.tsv"), sep="\t", index_col=0
)
MTB_df_diseased = pd.read_csv(
    os.path.join(folder_path, "MTB_df_diseased.tsv"), sep="\t", index_col=0
)
y_diseased = pd.read_csv(
    os.path.join(folder_path, "y_diseased.tsv"), sep="\t", index_col=0
)["target"]

# MP
MGX_df_MP = pd.read_csv(
    os.path.join(folder_path, "MGX_df_MP.tsv"), sep="\t", index_col=0
)
MTB_df_MP = pd.read_csv(
    os.path.join(folder_path, "MTB_df_MP.tsv"), sep="\t", index_col=0
)
y_MP = pd.read_csv(
    os.path.join(folder_path, "y_MP.tsv"), sep="\t", index_col=0
)["target"]

# Early
MGX_df_early = pd.read_csv(
    os.path.join(folder_path, "MGX_df_early.tsv"), sep="\t", index_col=0
)
MTB_df_early = pd.read_csv(
    os.path.join(folder_path, "MTB_df_early.tsv"), sep="\t", index_col=0
)
y_early = pd.read_csv(
    os.path.join(folder_path, "y_early.tsv"), sep="\t", index_col=0
)["target"]

# Intermediate
MGX_df_intermediate = pd.read_csv(
    os.path.join(folder_path, "MGX_df_intermediate.tsv"), sep="\t", index_col=0
)
MTB_df_intermediate = pd.read_csv(
    os.path.join(folder_path, "MTB_df_intermediate.tsv"), sep="\t", index_col=0
)
y_intermediate = pd.read_csv(
    os.path.join(folder_path, "y_intermediate.tsv"), sep="\t", index_col=0
)["target"]

# Late
MGX_df_late = pd.read_csv(
    os.path.join(folder_path, "MGX_df_late.tsv"), sep="\t", index_col=0
)
MTB_df_late = pd.read_csv(
    os.path.join(folder_path, "MTB_df_late.tsv"), sep="\t", index_col=0
)
y_late = pd.read_csv(
    os.path.join(folder_path, "y_late.tsv"), sep="\t", index_col=0
)["target"]

Study.Group
Healthy         157
Stage_I_II       69
Stage_III_IV     54
MP               40
Stage_0          27
Name: count, dtype: int64


In [13]:
#SVM model
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=63)

y_multiclass = meta_df["Study.Group"]

X_mgx = MGX_df_diseased  # Transformed CLR
X_mtb = MTB_df_diseased  # Transformed Log10

svm_mgx_pipe = make_pipeline(
    StandardScaler(),
    SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=63)
)

svm_mtb_pipe = make_pipeline(
    StandardScaler(),
    SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=63)
)

svm_mgx_auc = cross_val_score(
    svm_mgx_pipe, X_mgx, y_multiclass, cv=cv, scoring="roc_auc_ovr", n_jobs=-1
)
svm_mtb_auc = cross_val_score(
    svm_mtb_pipe, X_mtb, y_multiclass, cv=cv, scoring="roc_auc_ovr", n_jobs=-1
)

mgx_acc_scores = cross_val_score(
    svm_mgx_pipe, X_mgx, y_multiclass, cv=cv, scoring="accuracy", n_jobs=-1
)
mtb_acc_scores = cross_val_score(
    svm_mtb_pipe, X_mtb, y_multiclass, cv=cv, scoring="accuracy", n_jobs=-1
)
print("--- Multi Class (5-Fold Cross-Validation) ---")
print(f"SVM - Mean CV AUC (MGX): {svm_mgx_auc.mean():.4f} (± {svm_mgx_auc.std():.4f}) | Accuracy: {mgx_acc_scores.mean():.4f}")
print(f"SVM - Mean CV AUC (MTB): {svm_mtb_auc.mean():.4f} (± {svm_mtb_auc.std():.4f}) | Accuracy: {mtb_acc_scores.mean():.4f}")


--- Multi Class (5-Fold Cross-Validation) ---
SVM - Mean CV AUC (MGX): 0.4877 (± 0.0298) | Accuracy: 0.4296
SVM - Mean CV AUC (MTB): 0.6339 (± 0.0287) | Accuracy: 0.4296


In [14]:
#SVM model healthy vs MP

X_mgx = MGX_df_MP
X_mtb = MTB_df_MP  

svm_mgx_auc = cross_val_score(
    svm_mgx_pipe, X_mgx, y_MP, cv=cv, scoring="roc_auc_ovr", n_jobs=-1
)
svm_mtb_auc = cross_val_score(
    svm_mtb_pipe, X_mtb, y_MP, cv=cv, scoring="roc_auc_ovr", n_jobs=-1
)

mgx_acc_scores = cross_val_score(
    svm_mgx_pipe, X_mgx, y_MP, cv=cv, scoring="accuracy", n_jobs=-1
)
mtb_acc_scores = cross_val_score(
    svm_mtb_pipe, X_mtb, y_MP, cv=cv, scoring="accuracy", n_jobs=-1
)

print("--- MP (5-Fold Cross-Validation) ---")
print(f"SVM - Mean CV AUC (MGX): {svm_mgx_auc.mean():.4f} (± {svm_mgx_auc.std():.4f}) | Accuracy: {mgx_acc_scores.mean():.4f}")
print(f"SVM - Mean CV AUC (MTB): {svm_mtb_auc.mean():.4f} (± {svm_mtb_auc.std():.4f}) | Accuracy: {mtb_acc_scores.mean():.4f}")


--- MP (5-Fold Cross-Validation) ---
SVM - Mean CV AUC (MGX): 0.5068 (± 0.0774) | Accuracy: 0.7969
SVM - Mean CV AUC (MTB): 0.4554 (± 0.1554) | Accuracy: 0.7921


In [15]:
#SVM model healthy vs early

X_mgx = MGX_df_early
X_mtb = MTB_df_early  

svm_mgx_auc = cross_val_score(
    svm_mgx_pipe, X_mgx, y_early, cv=cv, scoring="roc_auc_ovr", n_jobs=-1
)
svm_mtb_auc = cross_val_score(
    svm_mtb_pipe, X_mtb, y_early, cv=cv, scoring="roc_auc_ovr", n_jobs=-1
)

mgx_acc_scores = cross_val_score(
    svm_mgx_pipe, X_mgx, y_early, cv=cv, scoring="accuracy", n_jobs=-1
)
mtb_acc_scores = cross_val_score(
    svm_mtb_pipe, X_mtb, y_early, cv=cv, scoring="accuracy", n_jobs=-1
)

print("--- early (5-Fold Cross-Validation) ---")
print(f"SVM - Mean CV AUC (MGX): {svm_mgx_auc.mean():.4f} (± {svm_mgx_auc.std():.4f}) | Accuracy: {mgx_acc_scores.mean():.4f}")
print(f"SVM - Mean CV AUC (MTB): {svm_mtb_auc.mean():.4f} (± {svm_mtb_auc.std():.4f}) | Accuracy: {mtb_acc_scores.mean():.4f}")


--- early (5-Fold Cross-Validation) ---
SVM - Mean CV AUC (MGX): 0.4374 (± 0.0806) | Accuracy: 0.8533
SVM - Mean CV AUC (MTB): 0.5328 (± 0.1943) | Accuracy: 0.8589


In [16]:
#SVM model healthy vs intermediate

X_mgx = MGX_df_intermediate
X_mtb = MTB_df_intermediate  

svm_mgx_auc = cross_val_score(
    svm_mgx_pipe, X_mgx, y_intermediate, cv=cv, scoring="roc_auc_ovr", n_jobs=-1
)
svm_mtb_auc = cross_val_score(
    svm_mtb_pipe, X_mtb, y_intermediate, cv=cv, scoring="roc_auc_ovr", n_jobs=-1
)

mgx_acc_scores = cross_val_score(
    svm_mgx_pipe, X_mgx, y_intermediate, cv=cv, scoring="accuracy", n_jobs=-1
)
mtb_acc_scores = cross_val_score(
    svm_mtb_pipe, X_mtb, y_intermediate, cv=cv, scoring="accuracy", n_jobs=-1
)

print("--- Intermediate (5-Fold Cross-Validation) ---")
print(f"SVM - Mean CV AUC (MGX): {svm_mgx_auc.mean():.4f} (± {svm_mgx_auc.std():.4f}) | Accuracy: {mgx_acc_scores.mean():.4f}")
print(f"SVM - Mean CV AUC (MTB): {svm_mtb_auc.mean():.4f} (± {svm_mtb_auc.std():.4f}) | Accuracy: {mtb_acc_scores.mean():.4f}")


--- Intermediate (5-Fold Cross-Validation) ---
SVM - Mean CV AUC (MGX): 0.4675 (± 0.0934) | Accuracy: 0.7036
SVM - Mean CV AUC (MTB): 0.7007 (± 0.0749) | Accuracy: 0.7080


In [17]:
#SVM model healthy vs late


X_mgx = MGX_df_late
X_mtb = MTB_df_late  

svm_mgx_auc = cross_val_score(
    svm_mgx_pipe, X_mgx, y_late, cv=cv, scoring="roc_auc_ovr", n_jobs=-1
)
svm_mtb_auc = cross_val_score(
    svm_mtb_pipe, X_mtb, y_late, cv=cv, scoring="roc_auc_ovr", n_jobs=-1
)

mgx_acc_scores = cross_val_score(
    svm_mgx_pipe, X_mgx, y_late, cv=cv, scoring="accuracy", n_jobs=-1
)
mtb_acc_scores = cross_val_score(
    svm_mtb_pipe, X_mtb, y_late, cv=cv, scoring="accuracy", n_jobs=-1
)

print("--- Late (5-Fold Cross-Validation) ---")
print(f"SVM - Mean CV AUC (MGX): {svm_mgx_auc.mean():.4f} (± {svm_mgx_auc.std():.4f}) | Accuracy: {mgx_acc_scores.mean():.4f}")
print(f"SVM - Mean CV AUC (MTB): {svm_mtb_auc.mean():.4f} (± {svm_mtb_auc.std():.4f}) | Accuracy: {mtb_acc_scores.mean():.4f}")


--- Late (5-Fold Cross-Validation) ---
SVM - Mean CV AUC (MGX): 0.4536 (± 0.0831) | Accuracy: 0.7393
SVM - Mean CV AUC (MTB): 0.7283 (± 0.0503) | Accuracy: 0.7680
